In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [3]:
Clean_Data_Path = Path(r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\02_Clean_Data")
Table_Path = Path(r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\tables")
Prediction_Path = Path(r"D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\predictions")

print("Clean data:", Clean_Data_Path)
print("Output tables:", Table_Path)
print("Output charts:", Prediction_Path)

Clean data: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\02_Clean_Data
Output tables: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\tables
Output charts: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\predictions


In [4]:
customers = pd.read_csv(
    Clean_Data_Path / "customers_clean.csv"
)

orders = pd.read_csv(
    Clean_Data_Path / "orders_clean.csv"
)

order_items = pd.read_csv(
    Clean_Data_Path / "order_items_clean.csv"
)

reviews = pd.read_csv(
    Clean_Data_Path / "reviews_clean.csv"
)

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Reviews:", reviews.shape)

Customers: (99441, 5)
Orders: (96461, 8)
Order items: (112650, 7)
Reviews: (99224, 7)


In [5]:
#Creating order revenue
order_revenue = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_revenue=("price", "sum")
    )
)

order_revenue.head()

,order_id,order_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90


In [6]:
#Building customer order history
customer_orders = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

customer_orders = customer_orders.merge(
    order_revenue,
    on="order_id",
    how="left"
)

customer_orders["order_revenue"] = (
    customer_orders["order_revenue"]
    .fillna(0)
)

customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,order_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,19.90


In [7]:
#Dataset reference date
reference_date = (
    customer_orders["order_purchase_timestamp"]
    .max()
)

print("Reference date:", reference_date)

Reference date: 2018-08-29 15:00:37


In [8]:
#Customer behavioral features
customer_features = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        last_purchase_date=(
            "order_purchase_timestamp",
            "max"
        ),

        first_purchase_date=(
            "order_purchase_timestamp",
            "min"
        ),

        frequency=(
            "order_id",
            "nunique"
        ),

        monetary_value=(
            "order_revenue",
            "sum"
        )
    )
    .reset_index()
)

In [9]:
#Feature calculations
customer_features["recency_days"] = (
    reference_date
    - customer_features["last_purchase_date"]
).dt.days

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase_date"]
    - customer_features["first_purchase_date"]
).dt.days

customer_features["average_order_value"] = np.where(
    customer_features["frequency"] > 0,
    customer_features["monetary_value"]
    / customer_features["frequency"],
    0
)

customer_features.head()

,customer_unique_id,last_purchase_date,first_purchase_date,frequency,monetary_value,recency_days,customer_lifetime_days,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,129.90,111,0,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,18.90,114,0,18.90
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,69.00,536,0,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,25.99,320,0,25.99
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,180.00,287,0,180.00


In [10]:
#Define inactivity risk
INACTIVITY_THRESHOLD_DAYS = 180

customer_features["inactivity_risk"] = (
    customer_features["recency_days"]
    > INACTIVITY_THRESHOLD_DAYS
).astype(int)

customer_features[
    "inactivity_risk"
].value_counts()

inactivity_risk
1    54996
0    38346
Name: count, dtype: int64

In [11]:
#Checking class distribution,    This is important because we need to know whether the target is heavily imbalanced.
risk_distribution = pd.DataFrame({
    "inactivity_risk": [
        0,
        1
    ],
    "customer_count": [
        (
            customer_features["inactivity_risk"] == 0
        ).sum(),

        (
            customer_features["inactivity_risk"] == 1
        ).sum()
    ]
})

risk_distribution["percentage"] = (
    risk_distribution["customer_count"]
    / len(customer_features)
    * 100
)

risk_distribution

,inactivity_risk,customer_count,percentage
0,0,38346,41.081185
1,1,54996,58.918815


In [12]:
#Selecting ML features
feature_columns = [
    "frequency",
    "monetary_value",
    "average_order_value",
    "customer_lifetime_days"
]

X = customer_features[
    feature_columns
].copy()

y = customer_features[
    "inactivity_risk"
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (93342, 4)
y shape: (93342,)


In [13]:
#Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 74673
Testing rows: 18669


In [14]:
#Train Random Forest classifier
churn_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

churn_model.fit(
    X_train,
    y_train
)

print("Inactivity-risk model trained.")

Inactivity-risk model trained.


In [15]:
#Predictions
y_pred = churn_model.predict(
    X_test
)

y_probability = churn_model.predict_proba(
    X_test
)[:, 1]

In [16]:
#Classification report
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.48      0.61      0.54      7669
           1       0.67      0.55      0.60     11000

    accuracy                           0.57     18669
   macro avg       0.58      0.58      0.57     18669
weighted avg       0.59      0.57      0.58     18669



In [17]:
#Confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4673 2996]
 [4982 6018]]


In [18]:
#ROC-AUC
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print(
    "ROC-AUC:",
    round(roc_auc, 4)
)

ROC-AUC: 0.6105


In [44]:
roc_auc_summary = pd.DataFrame({
    "metric": ["ROC-AUC"],
    "value": [round(roc_auc, 4)]
})

roc_auc_summary.to_csv(
    Prediction_Path / "customer_risk_model_performance.csv",
    index=False
)

In [34]:
#Feature importance
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": churn_model.feature_importances_,
}).sort_values( 
    "importance",
    ascending=False,
)

feature_importance

,feature,importance
2,average_order_value,0.445397
1,monetary_value,0.427045
3,customer_lifetime_days,0.112289
0,frequency,0.015268


In [20]:
feature_importance.to_csv(
    Table_Path / "customer_inactivity_feature_importance.csv",
    index=False
)

print(
    "Saved:",
    Table_Path
    / "customer_inactivity_feature_importance.csv"
)

Saved: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\tables\customer_inactivity_feature_importance.csv


In [21]:
#Generate risk scores for all customers
customer_features["inactivity_probability"] = (
    churn_model.predict_proba(
        customer_features[feature_columns]
    )[:, 1]
)

customer_features["risk_level"] = pd.cut(
    customer_features["inactivity_probability"],
    bins=[
        -np.inf,
        0.33,
        0.66,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

customer_features[
    [
        "customer_unique_id",
        "recency_days",
        "frequency",
        "monetary_value",
        "inactivity_probability",
        "risk_level"
    ]
].head(10)

,customer_unique_id,recency_days,frequency,monetary_value,inactivity_probability,risk_level
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,129.90,0.510721,Medium
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,18.90,0.505989,Medium
2,0000f46a3911fa3c0805444483337064,536,1,69.00,0.507264,Medium
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,25.99,0.512187,Medium
4,0004aac84e0df4da2b147fca70cf8255,287,1,180.00,0.492848,Medium
5,0004bd2a26a76fe21f786e4fbd80607f,145,1,154.00,0.484945,Medium
6,00050ab1314c0e55a6ca13cf7181fecf,131,1,27.99,0.498278,Medium
7,00053a61a98854899e70ed204dd4bafe,182,1,382.00,0.508431,Medium
8,0005e1862207bf6ccc02e4228effd9a0,542,1,135.00,0.432624,Medium
9,0005ef4cd20d2893f0d9fbd94d3c0d97,169,1,104.90,0.541354,Medium


In [43]:
#High-risk customer summary
risk_summary = (
    customer_features
    .groupby(
        "risk_level",
        observed=True
    )
    .agg(
        customers=(
            "customer_unique_id",
            "nunique"
        ),

        average_recency_days=(
            "recency_days",
            "mean"
        ),

        average_frequency=(
            "frequency",
            "mean"
        ),

        average_monetary_value=(
            "monetary_value",
            "mean"
        ),

        average_inactivity_probability=(
            "inactivity_probability",
            "mean"
        )
    )
    .reset_index()
)

risk_summary

,risk_level,customers,average_recency_days,average_frequency,average_monetary_value,average_inactivity_probability
0,Low,2612,140.323507,1.232772,114.311677,0.267781
1,Medium,86154,236.092869,1.024062,145.368331,0.498812
2,High,4576,307.786058,1.095717,86.625966,0.719890


In [23]:
risk_summary.to_csv(
    Table_Path / "customer_inactivity_risk_summary.csv",
    index=False
)

print(
    "Saved:",
    Table_Path
    / "customer_inactivity_risk_summary.csv"
)

Saved: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\tables\customer_inactivity_risk_summary.csv


In [24]:
customer_risk_output = customer_features[
    [
        "customer_unique_id",
        "first_purchase_date",
        "last_purchase_date",
        "recency_days",
        "frequency",
        "monetary_value",
        "average_order_value",
        "customer_lifetime_days",
        "inactivity_risk",
        "inactivity_probability",
        "risk_level"
    ]
].copy()

customer_risk_output.to_csv(
    Prediction_Path
    / "customer_inactivity_risk_predictions.csv",
    index=False
)

print(
    "Saved:",
    Prediction_Path
    / "customer_inactivity_risk_predictions.csv"
)

Saved: D:\Projects\Smart_E-Commerce_Demand_&_Operations_Intelligence_System\09_Outputs\predictions\customer_inactivity_risk_predictions.csv


In [25]:
print("Customers analyzed:", len(customer_risk_output))

print("\nRisk levels:")
print(
    customer_risk_output["risk_level"]
    .value_counts()
)

print("\nInactivity target:")
print(
    customer_risk_output["inactivity_risk"]
    .value_counts()
)

print("\nROC-AUC:", round(roc_auc, 4))

Customers analyzed: 93342

Risk levels:
risk_level
Medium    86154
High       4576
Low        2612
Name: count, dtype: int64

Inactivity target:
inactivity_risk
1    54996
0    38346
Name: count, dtype: int64

ROC-AUC: 0.6105
